## Módulo de limpieza de datasets de Fondos.

-> Los datasets leidos y limpiados en este módulo provienen del scraping de la siguiente página Web:
- [Morningstar](https://www.morningstar.es/es/)

A continuación, para la limpieza de datos seguiremos los siguientes pasos.
1. Lectura de datos y concatenación de dataframes.
2. Eliminado de filas duplicadas.
3. Tratamiento de celdas
---

In [ ]:
import pandas as pd

### 1. Limpieza de datos y concatenación de dataframes.

In [ ]:
# Valores que significan nulos en los excel.
na_val = {'-', 'n/a'}

# Lectura de los excel y declaración de nulos como np.nan
df1 = pd.read_excel('../data/raw/DatosFondos1.xlsx', na_values=na_val, index_col=0)
df2 = pd.read_excel('../data/raw/DatosFondos2.xlsx', na_values=na_val, index_col=0)
df3 = pd.read_excel('../data/raw/DatosFondos3.xlsx', na_values=na_val, index_col=0)

# Concatenamos los datasets.
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# Para ver el número de filas
df.tail() 


### 2. Eliminado de filas que no interesan.

- Eliminamos las filas cuyos ISIN se repiten (identificador)

In [ ]:
df = df.drop_duplicates(subset=['ISIN'], keep='first') # Si 2 se repiten, conservamos el primero
df.head()

- Eliminamos filas que no tienen datos en Gastos Corrientes.

Esto lo hacemos porque sin ese dato no podemos calcular la rentabilidad neta, y inventarselo supondría **pérdida de fiabilidad**

In [ ]:
df = df.dropna(subset=['Gastos Corrientes'])
df.head()

- Estudiemos nulos por columnas.

In [ ]:
df.isnull().sum() # sumamos nulos por columnas

- Como queremos calcular rentabilidades desde 2017, eliminemos filas que no tienen datos en esa fecha y volvamos a estudiar qué pasa.

In [ ]:
df = df.dropna(subset=['rent 2017'])
df.head()

- Si observamos bien, hemos eliminado todos los datos de rentabilidad nulos, ya que si tenían nulos en 2018 (por ejemplo) implicaba que también los tuviesen en 2017.

In [ ]:
# Cantidad de nulos por columna
df.isnull().sum() # solo quedan 5 en gastos de gestión max anual (columna que no usaremos)


- Una vez eliminadas las filas innecesarias, reseteamos índices para evitar problemas con filas.

In [ ]:
df.reset_index(drop=True, inplace=True)
df.tail() #159 filas resultantes

### 3. Tratamiento de celdas

Ahora vamos a dejar las celdas disponibles para poder operar con ellas.

- Crearemos una función que lo haga directamente.

In [ ]:
def limpiador(dato) -> float:
    ''' Elimina caracteres innecesarios y transforma a float '''
    
    # Hago los cambios uno a uno ya que no me deja pasarle un diccionario de cambios
    dato =str(dato)
    sin_coma = dato.replace(',', '.')
    sin_porcentaje = sin_coma.replace('%', '')
    sin_espacios = sin_porcentaje.replace(' ', '')
    return float(sin_espacios)

- Aplicamos la función a las columnas que nos interesan.


In [ ]:
df.loc[:, 'rent 2024':'Volatilidad'] = df.loc[:, 'rent 2024':'Volatilidad'].map(limpiador)

# pasamos a formato decimal las columnas de rentabilidad, comisiones y volatilidad
df.loc[:, 'rent 2024':'Volatilidad'] = df.loc[:, 'rent 2024':'Volatilidad']/100
df.head()

---
Una vez terminada la limpieza, extraemos el dataframe final.

In [ ]:
df.to_excel('../data/processed/Excel_final_fondos.xlsx')